# LeaseGuard Phase 6 — Qwen3.5-4B T4 baseline

This Google Colab **T4** notebook evaluates one unmodified publisher checkpoint on the frozen Dataset v1 evaluation split.

It is a thin caller of `src/leaseguard`. Do not paste lease text, secrets, or prediction files back into Git.

Runtime: **T4 GPU**. Disk: mount Drive so checkpoints survive disconnects.

Qwen3.5-4B is Apache-2.0 and should fit T4 in 4-bit. It is the only candidate that also runs the 8-bit comparison.

Comparisons run in a fixed order:

1. `direct-full-4bit`
2. `schema-full-4bit` (required for model selection)
3. `schema-clause-4bit`
4. `schema-retrieval-4bit`
5. `schema-full-8bit` only for Qwen3.5-4B

If a session disconnects, rerun from the top. Completed examples resume from Drive checkpoints.


In [ ]:
import os
from pathlib import Path

CANDIDATE = "qwen35-4b"
REPO_URL = "https://github.com/YOUR_USER/LeaseGuard.git"
REPO_DIR = Path("/content/LeaseGuard")
DRIVE_ROOT = Path("/content/drive/MyDrive/leaseguard")
REPORT_DIR = DRIVE_ROOT / "reports" / "baselines"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "baselines"
EXAMPLES_ROOT = DRIVE_ROOT / "dataset_examples"
HF_TOKEN = ""  # required for gated Gemma; leave empty for Qwen
COMPARISONS = None  # None = every comparison allowed for this candidate
LIMIT = None  # set to 1 for a smoke test

print({"candidate": CANDIDATE, "drive": str(DRIVE_ROOT)})

In [ ]:
import sys
from subprocess import check_call, run

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; using paths already set above.")

gpu = run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if "T4" not in (gpu.stdout or "") and "Tesla T4" not in (gpu.stdout or ""):
    print("Warning: this notebook expects a Tesla T4. Runtime > Change runtime type > T4 GPU.")

if not REPO_DIR.exists():
    check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "accelerate",
        "bitsandbytes",
        "huggingface_hub",
    ]
)
for path in (REPORT_DIR, CHECKPOINT_DIR, EXAMPLES_ROOT):
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
from leaseguard.evaluation.baseline import get_candidate, load_baseline_registry

os.environ["LEASEGUARD_ALLOW_MODEL_DOWNLOAD"] = "1"
registry = load_baseline_registry()
candidate = get_candidate(registry, CANDIDATE)
print(candidate.model_id, candidate.license, "gated=", candidate.gated)

if candidate.gated:
    if not HF_TOKEN:
        raise SystemExit("Gated model: accept the Hugging Face license, then set HF_TOKEN.")
    from huggingface_hub import login

    login(token=HF_TOKEN)

In [ ]:
from leaseguard.evaluation.cli import main as evaluation_main

examples = (
    EXAMPLES_ROOT
    if any(EXAMPLES_ROOT.rglob("*.json"))
    else REPO_DIR / "data" / "samples" / "dataset_v1" / "examples"
)
allowed = [item.comparison_id for item in registry.comparisons if CANDIDATE in item.candidate_ids]
selected = COMPARISONS or allowed
print("comparisons", selected)

for comparison_id in selected:
    argv = [
        "baseline",
        "--candidate",
        CANDIDATE,
        "--comparison",
        comparison_id,
        "--backend",
        "transformers",
        "--allow-download",
        "--examples",
        str(examples),
        "--output-dir",
        str(REPORT_DIR),
        "--checkpoint-dir",
        str(CHECKPOINT_DIR),
        "--hardware",
        "Google Colab T4",
    ]
    if LIMIT is not None:
        argv.extend(["--limit", str(LIMIT)])
    print("\n===", comparison_id, "===")
    status = evaluation_main(argv)
    print("exit", status)
    if status != 0:
        print("Comparison failed; later comparisons still run so a T4 OOM can be recorded.")

In [ ]:
from pathlib import Path

print("Saved runs:")
for path in sorted(REPORT_DIR.glob(f"{CANDIDATE}-*.json")):
    print(path)

## After this candidate finishes

Run the other two candidate notebooks, then open `colab_baseline_compare.ipynb` (CPU is enough) to select the strongest practical T4 model.

Official LegalBench / CUAD / ContractNLI / LegalBench-RAG scoring still uses `colab_orchestrator.ipynb` with `TASK=evaluate-*` after this product-task matrix. Those scores are the only headline lab numbers; these baseline JSON files are not a blended LeaseGuard score.
